Now we need to perform the market-relative features calculations across all 21 coins.

Import necessary libraries.

In [1]:
import os
import pandas as pd
import numpy as np

Now load the data from `unnormalized-feature-engineering` and stitch all the seperate coins into a master CSV.

In [2]:
dataframes = []

for file in os.listdir("unnormalized-feature-engineering"):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join("unnormalized-feature-engineering", file))
        dataframes.append(df)

master_df = pd.concat(dataframes, ignore_index=True)

master_df["open_time"] = pd.to_datetime(master_df["open_time"])
master_df = master_df.sort_values(by=["open_time", "symbol"]).reset_index(drop=True)

Now create the market-relative feature engineering function.

In [3]:
EPSILON = 1e-8

def construct_market_relative_features(df):
    df["mkt_log_ret_1"] = df.groupby("open_time")["log_ret_1d"].transform("mean")

    df["ret_vs_mkt_1"] = df["log_ret_1d"] - df["mkt_log_ret_1"]

    df["mkt_rvol_21"] = df.groupby("open_time")["rvol_21"].transform("mean")

    df["rel_vol"] = df["rvol_21"] / (df["mkt_rvol_21"] + EPSILON)

    # Market relative features should not be added until 2017-12-12
    # see the first candle timestamp notebook for reasoning on why this date in particular was chosen
    threshold_date = "2017-12-12"
    market_cols = ["mkt_log_ret_1", "ret_vs_mkt_1", "mkt_rvol_21", "rel_vol"]

    df.loc[df["open_time"] < threshold_date, market_cols] = np.nan

    return df

Call on `master_df`.

In [4]:
market_master_df = construct_market_relative_features(master_df)

In [5]:
print(market_master_df.head())

                  open_time     open     high      low    close      volume  \
0 2017-08-17 04:00:00+00:00  4261.48  4313.62  4261.32  4308.83   47.181009   
1 2017-08-17 04:00:00+00:00   301.13   302.57   298.00   301.61  125.668770   
2 2017-08-17 05:00:00+00:00  4308.83  4328.69  4291.37  4315.32   23.234916   
3 2017-08-17 05:00:00+00:00   301.61   303.28   300.00   303.10  377.672460   
4 2017-08-17 06:00:00+00:00  4330.29  4345.45  4309.37  4324.35    7.229691   

    symbol  log_ret_1d  log_ret_5d  log_ret_21d  ...  log_volume  \
0  BTCUSDT         NaN         NaN          NaN  ...    3.874965   
1  ETHUSDT         NaN         NaN          NaN  ...    4.841576   
2  BTCUSDT         NaN         NaN          NaN  ...    3.187794   
3  ETHUSDT         NaN         NaN          NaN  ...    5.936672   
4  BTCUSDT         NaN         NaN          NaN  ...    2.107748   

   log_dollar_volume  rel_dvol_21  dvol_z_21  amihud_illiq_1  amihud_illiq_21  \
0          12.222418          NaN  

Make test CSV export

In [ ]:
# pd.DataFrame.to_csv(market_master_df.loc[market_master_df['symbol'] == "DGBUSDT"], "DGBUSDT-market-relative-test.csv", index=False)

In [7]:
print(f"Number of rows in the master dataframe: {len(master_df)}")

Number of rows in the master dataframe: 1435478


Export master dataframe.

NOTE: This still must go through normalization and train/eval/test split

In [8]:
pd.DataFrame.to_csv(market_master_df, "market-master-df.csv", index=False)